In [15]:
## Reading the dataset
data = open('input.txt','r',encoding='utf-8').read()

In [100]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
eval_interval = 300
learning_rate = 1e-3
max_iters = 5000
n_embed = 32

In [16]:
len(data)

1115394

In [17]:
## first few thousands of characters
print(data[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [101]:
chars = sorted(list(set(data)))
char_size = len(chars)
print(''.join(chars))
print(char_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [102]:
vocab_size = char_size

In [103]:
## First thing (Think how will you create tokens from the dataset) (as this one is a character level model so we need to represent them in number)
## building encoder (Take string and give out integer) , decoder (Take integer and give out string)

stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for s, i in stoi.items()}
encoder = lambda s : [stoi[c] for c in s]
decoder = lambda i : ''.join([itos[c] for c in i])

print(encoder('hi, there'))
print(decoder(encoder('hi, there')))

[46, 47, 6, 1, 58, 46, 43, 56, 43]
hi, there


In [120]:
## encoding the entire dataset
import torch
encoData = torch.tensor(encoder(data), dtype=torch.long)
print(encoData.shape , encoData.type)
print(encoData[:1000])

torch.Size([1115394]) <built-in method type of Tensor object at 0x119a63c50>
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15,

In [121]:
## splitting dataset for train and validation
n = int(0.9*len(encoData))

train_data = encoData[:n]
val_data = encoData[n:]

print(train.shape)
print(val.shape)

torch.Size([1003854])
torch.Size([111540])


In [122]:
context_length = 8
encoData[:context_length+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [123]:
## Processing data in batch
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_length, (batch_size,))
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    ## --- for implementing transformer ---
    x , y = x.to(device) , y.to(device)
    return x,y

xb , yb = get_batch('train')
print(f'inputs:{xb.shape}')
print(f'outputs:{yb.shape}')

inputs:torch.Size([4, 8])
outputs:torch.Size([4, 8])


In [165]:
class Block(nn.Module):
    # -- Transformer Block : communication followed by computation -- #
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)
    def forward(self, x):
        x = x + self.sa(x)
        x = x + self.ffwd(x)
        return x

In [163]:
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embed, n_embed), nn.ReLU(), nn.Linear(n_embed, n_embed))
    def forward(self,x):
        return self.net(x)



In [162]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)      # (B,T,C)
        q = self.query(x)    # (B,T,C)

        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * C**-0.5  # (B,T,C) @ (B,C,T) -> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # (B,T,T)
        wei = F.softmax(wei, dim=-1)  # (B,T,T)

        # perform the weighted aggregation of the values
        v = self.value(x)  # (B,T,C)
        out = wei @ v  # (B,T,T) @ (B,T,C) -> (B,T,C)

        return out

In [164]:
class MultiHeadAttention(nn.Module):
    # ---- multihead of self attention in parallel ---- #
    def __init__(self, nums_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(nums_head)])
        self.proj = nn.Linear(n_embed, n_embed)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out


In [155]:
## implementing biagram model
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BiagramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.ln_head = nn.Linear(n_embed, vocab_size)
        # self.sa_head = MultiHeadAttention(4, n_embed//4)
        # self.ffwd = FeedForward(n_embed)
        self.blocks = nn.Sequential(Block(n_embed, n_head=4), Block(n_embed, n_head=4), Block(n_embed, n_head=4))
        ## positional embedding
        self.positional_embed = nn.Embedding(block_size, n_embed)
    def forward(self, idx, target=None):
        B, T = idx.shape
        token_embed = self.token_embedding_table(idx)
        positions = self.positional_embed(torch.arange(T, device=device))
        x = token_embed + positions
        # x = self.sa_head(x)
        # x = self.ffwd(x)
        x = self.blocks(x)
        logits = self.ln_head(x)
        ## loss
        if target == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T,C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # Focus on last time steps
            # softmwax
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BiagramLanguageModel()
m = model.to(device)
out, loss = m.forward(xb, yb)
print(out.shape)
print(loss)

torch.Size([32, 65])
tensor(4.2144, grad_fn=<NllLossBackward0>)


In [156]:
print(decoder(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


cZCm
$-ixXGbeiGAlXH!$fxnohFRE
Pqcs,u'cK;UMSwJziSXniZhMeQGs
QBNbJ&R$j;bGMHyStdx?ho.Zuks V$Iu$d3QsXmg!


In [157]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [158]:
# created a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

In [160]:

for iter in range(max_iters):
    ## --- loss estimation 
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f'step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}')
    xb, yb = get_batch('train')

    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
# print(loss.item())

step 0: train loss 2.6037, val loss 2.6005
step 300: train loss 2.5960, val loss 2.5454
step 600: train loss 2.6156, val loss 2.5601
step 900: train loss 2.5560, val loss 2.5768
step 1200: train loss 2.5649, val loss 2.5407
step 1500: train loss 2.5680, val loss 2.5475
step 1800: train loss 2.5293, val loss 2.5308
step 2100: train loss 2.5455, val loss 2.5211
step 2400: train loss 2.4994, val loss 2.5125
step 2700: train loss 2.5637, val loss 2.5437
step 3000: train loss 2.5001, val loss 2.4974
step 3300: train loss 2.4931, val loss 2.4764
step 3600: train loss 2.4750, val loss 2.4751
step 3900: train loss 2.4974, val loss 2.4658
step 4200: train loss 2.4403, val loss 2.4819
step 4500: train loss 2.4606, val loss 2.4660
step 4800: train loss 2.4369, val loss 2.4594


In [161]:
print(decoder(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


NAMEN:
Gy Rm
Metett Rmk bare ols tolte yo
Pelor 's prees asbll, as thy le goorte ce-or ceo ker hay that ang Tolg;
Rhe lit delk,
Ener tis to fottatot sthist an
Wet,
Han thuf ro, co ist hilg ceark,
thher bed thit u matirc hellat wrot find
Pond,
Ofd o ad, N.E FASS:
Thn tle I sorle esit yof, hi ther olt


In [ ]:
## We are seeing some improvment from the last prediction to these new predictions, but not that good, because issue is tokens are not talking to each other
## No relationship between them, the model is only predicting next character based on previous 8 characters, so certainlly it is not good. Let build somthing so that
## Model can talk to each other


In [69]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)

torch.Size([2]) torch.Size([1, 2])
torch.Size([2]) torch.Size([2, 2])
torch.Size([2]) torch.Size([3, 2])
torch.Size([2]) torch.Size([4, 2])
torch.Size([2]) torch.Size([5, 2])
torch.Size([2]) torch.Size([6, 2])
torch.Size([2]) torch.Size([7, 2])
torch.Size([2]) torch.Size([8, 2])
torch.Size([2]) torch.Size([1, 2])
torch.Size([2]) torch.Size([2, 2])
torch.Size([2]) torch.Size([3, 2])
torch.Size([2]) torch.Size([4, 2])
torch.Size([2]) torch.Size([5, 2])
torch.Size([2]) torch.Size([6, 2])
torch.Size([2]) torch.Size([7, 2])
torch.Size([2]) torch.Size([8, 2])
torch.Size([2]) torch.Size([1, 2])
torch.Size([2]) torch.Size([2, 2])
torch.Size([2]) torch.Size([3, 2])
torch.Size([2]) torch.Size([4, 2])
torch.Size([2]) torch.Size([5, 2])
torch.Size([2]) torch.Size([6, 2])
torch.Size([2]) torch.Size([7, 2])
torch.Size([2]) torch.Size([8, 2])
torch.Size([2]) torch.Size([1, 2])
torch.Size([2]) torch.Size([2, 2])
torch.Size([2]) torch.Size([3, 2])
torch.Size([2]) torch.Size([4, 2])
torch.Size([2]) torc

In [ ]:
# Same as above but more efficient
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) (B, T, C) -> (B,T,C)
torch.allclose(xbow, xbow2)

True

In [ ]:
## same as above but in softmax version
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))

wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [88]:
## Self Attention
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x= torch.randn(B, T, C)

# single head self attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
wei = q @ k.transpose(-2, -1) # (B,T,16) * (B, 16, T) --> (B, T, T)
tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T,T))

wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
v = value(x)
out = wei @ v

out



tensor([[[-3.9045e-03,  2.1869e-02,  4.0135e-03, -1.9441e-02, -3.5507e-03,
           1.8556e-02,  2.5018e-03, -1.3019e-02, -2.2047e-02,  4.7380e-03,
           4.3773e-03, -1.4766e-02, -1.1958e-02, -1.2076e-02,  7.1124e-03,
           1.4188e-02],
         [ 6.7801e-03, -2.8829e-03, -2.1238e-03,  6.9400e-04, -1.8806e-03,
          -6.1069e-04, -4.3032e-03, -2.9935e-03, -3.6059e-03,  8.5297e-03,
           8.3296e-03, -3.3179e-03,  1.8705e-03,  2.0327e-03,  2.2261e-04,
           1.5746e-02],
         [ 1.5285e-01, -2.7215e-02, -1.4882e-01,  6.7228e-02,  7.3462e-02,
          -6.4763e-02,  3.0644e-02,  2.6180e-02, -7.7010e-02, -1.9796e-01,
           6.2980e-02,  8.4133e-02, -1.2460e-01, -3.7621e-02, -5.7744e-02,
           3.9307e-01],
         [ 1.4621e-01,  1.3626e-01, -1.2934e-01, -1.1850e-01,  1.1332e-01,
           5.1606e-02,  5.0257e-02, -1.0902e-01, -2.3622e-01, -3.7930e-02,
           1.9340e-02, -5.3485e-02, -1.5486e-01, -7.7516e-02,  1.4516e-01,
           4.4782e-01],
    

In [ ]:
# Attention is a communication mechanism. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
# There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
# Each example across batch dimension is of course processed completely independently and never "talk" to each other.
# In an "encoder" attention block just delete the single line that does masking with tril, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
# "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module).
# "Scaled" attention additional divides wei by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below.

In [87]:
wei

tensor([[[0.0248, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0052, 0.0091, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0521, 0.0135, 0.2482, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3171, 0.0214, 0.1642, 0.1188, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0412, 0.0487, 0.1046, 0.0742, 0.2000, 0.0000, 0.0000, 0.0000],
         [0.1060, 0.5347, 0.2059, 0.1030, 0.7402, 0.0192, 0.0000, 0.0000],
         [0.4298, 0.3409, 0.1769, 0.2027, 0.0480, 0.8472, 0.2329, 0.0000],
         [0.0238, 0.0316, 0.1002, 0.5013, 0.0117, 0.1336, 0.7671, 1.0000]],

        [[0.0443, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0042, 0.0375, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0560, 0.0210, 0.2496, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3679, 0.1441, 0.4929, 0.0438, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0088, 0.1052, 0.0604, 0.5847, 0.2046, 0.0000, 0.0000, 0.0000],
         [0.0367, 0.089